In [1]:

from pathlib import Path
import os


os.environ["HF_HOME"] = r"D:\Workspace\RAG_Integration\SamplePDF\hf_cache"
os.environ["ONEAPI_DEVICE_SELECTOR"] = "opencl:gpu"
os.environ["SYCL_DEVICE_FILTER"] = "gpu"
os.environ["CLI_RelaxAllocationLimits"] = "1"  # Fixes VRAM limits on Windows

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions, TableStructureOptions, smolvlm_picture_description
from docling.document_converter import DocumentConverter, WordFormatOption, PdfFormatOption
from dotenv import load_dotenv

D:\Workspace\RAG_Integration\RAG_System_Unified_Search_Engine\PDF_Extrating\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


You will need to install PyTorch with XPU support and the Intel Extension for PyTorch. Use the following pip command:

pip install --no-cache-dir torch torchvision torchaudio --index-url https://download.pytorch.org/whl/xpu

To check the availabe GPU

In [2]:
##os.environ["ZES_ENABLE_SYSMAN"] = "1"
#os.environ["ZE_AFFINITY_MASK"] = "0"          # Forces focus on primary Intel GPU
#os.environ["ONEAPI_DEVICE_SELECTOR"] = "opencl:gpu"
#os.environ["SYCL_DEVICE_FILTER"] = "gpu"
#os.environ["CLI_RelaxAllocationLimits"] = "1"

#import torch
#print("XPU Count detected by Windows:", torch._C._xpu_getDeviceCount() if hasattr(torch, '_C') else "No C binding")
#print("XPU Is Available:", torch.xpu.is_available())


XPU Is Available: False


In [2]:
load_dotenv()

False

Convert PDF to Markdown

Document converter can be used for PDF as well as DOCX.

In [3]:
table_structure_option= TableStructureOptions(
    do_cell_matching = True
)
pipeline_option = PdfPipelineOptions (
    generate_page_images=True,
    images_scale=1.00,
    do_ocr=True,
    do_picture_description=True,
    ocr_options=RapidOcrOptions(),
    do_table_structure=True,
    table_structure_options = table_structure_option,
    picture_description_options=smolvlm_picture_description
)

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_option
        ),
        InputFormat.DOCX: WordFormatOption(),   # default options
    }
)
document_path = Path("data/Sample.pdf")
document_path

WindowsPath('data/Sample.pdf')

In [4]:
%%time
result = converter.convert(document_path)
documents = result.document
print(documents.export_to_markdown(page_no=1))


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.
Loading weights: 100%|██████████| 471/471 [00:00<00:00, 2136.75it/s]
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
[INFO] 2026-07-26 16:44:25,305 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-26 16:44:25,717 [RapidOCR] download_file.py:60: File exists and is valid: D:\Workspace\RAG_Integration\RAG_System_Unified_Search_Engine\PDF_Extrating\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-26 16:44:25,719 [RapidOCR] main.py:63: Using D:\Workspace\RAG_Integration\RAG_System_Unified_Search_Engine\PDF_Extrating\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-07-

<!-- image -->

## NVIDIA Announces Financial Results for Fourth Quarter and Fiscal 2025

- Record quarterly revenue of $39.3 billion, up 12% from Q3 and up 78% from a year ago
- Record quarterly Data Center revenue of $35.6 billion, up 16% from Q3 and up 93% from a year ago
- Record full-year revenue of $130.5 billion, up 114%

NVIDIA (NASDAQ: NVDA) today reported revenue for the fourth quarter ended January 26, 2025, of $39.3 billion, up 12% from the previous quarter and up 78% from a year ago.

For the quarter, GAAP earnings per diluted share was $0.89, up 14% from the previous quarter and up 82% from a year ago. Non-GAAP earnings per diluted share was $0.89, up 10% from the previous quarter and up 71% from a year ago.

For fiscal 2025, revenue was $130.5 billion, up 114% from a year ago. GAAP earnings per diluted share was $2.94, up 147% from a year ago. Non-GAAP earnings per diluted share was $2.99, up 130% from a year ago.

'Demand for Blackwell is amazing as reasoning AI adds an

In [5]:
from functools import lru_cache
from docling_core.types.doc import DocItemLabel


class MetaData:
    def __init__(self, file_name:str, location:str, page_no:int):
        self.file_name =file_name
        self.location = location
        self.page_no = page_no

    def __str__(self):

        return (f" \"metadata\": "
                f"{{ "
                f"\"document_name\":\"{self.file_name}\", "
                f"\"file_location\":\"{self.location} \" }}")

class DataStructure:
    def __init__(self, obj_type:DocItemLabel, level:int, content:str, meta:MetaData):
        self.level = level
        self.content = content
        self.obj_type = obj_type
        self.meta = meta

    def __str__(self) -> str :

        return (f" {{ "
                f"\"content\":\"{self.content}\", "
                f"{self.meta} }}")



class DataLinkStructure(DataStructure):
    def __init__(self, obj_type:DocItemLabel, level:int, content:str, meta:MetaData, id = str):
        super().__init__(obj_type, level, content, meta)
        self._link_data = None
        self._section_data = None
        self.id = id

    @property
    def link_data(self):
        return self._link_data

    @link_data.setter
    def link_data(self, link_data:DataStructure = None):
        self._link_data = link_data

    @property
    def section_data(self):
        return self._section_data

    @section_data.setter
    def section_data(self, section_data = None):
        self._section_data = section_data

    @property
    def section(self) -> str:
        return self.cal_sub_section_data()[0]

    @property
    def sub_section_data(self) -> str:
        return self.cal_sub_section_data()[1]

    @lru_cache(maxsize=5)
    def cal_sub_section_data(self) -> list:
        section_data:str
        sub_section:list = []
        sub_section_data:str = ''
        section:DataLinkStructure = self._section_data
        while section is not None:
            sub_section.append(section.content)
            section = section._section_data

        if sub_section and len(sub_section) > 0:
            if self.obj_type == DocItemLabel.SECTION_HEADER:
                sub_section.insert(0, self.content)
            section_data = sub_section[-1] # Reteieve the last index which is parent
            del sub_section[-1] # Delete the last index
            sub_section_data = '->'.join(sub_section[::-1]) if sub_section else '' #Reverse the order
        else:
            section_data = self.content if self.obj_type == DocItemLabel.SECTION_HEADER else ''

        return [section_data, sub_section_data]

    def __str__(self):

        data = self.cal_sub_section_data()

        return (f" {{ "
                f"\"ID\" : \"{self.id}\","
                f"\"section\":\"{data[0]}\","
                f"\"sub_section\":\"{data[1]}\","
                f"\"content\":\"{self.content} \","
                f"\"level\":{self.level},"
                f"\"page\":{self.meta.page_no},"
                f"{self.meta} }}")

class TableObj(DataLinkStructure):
    def __init__(self, obj_type:DocItemLabel, level:int, text:str, meta:MetaData, summary:str):
        super().__init__(obj_type, level, text, meta)
        self.summary = summary



In [6]:
from docling_core.types.doc import SectionHeaderItem


def retrieve_chunk(node, doc_name:str, doc_path:str, obj_type, id) -> DataLinkStructure:

    if hasattr(node, 'prov') and node.prov:
        meta = MetaData(page_no=node.prov[0].page_no, file_name=doc_name, location=doc_path)
    else:
         meta = MetaData(page_no=0, file_name=doc_name, location=doc_path)

    return DataLinkStructure(obj_type=obj_type, level= node.level if hasattr(node, 'level') else 0, content=node.text, meta=meta, id = id)

def retrieve_table_chunk(node, doc_name:str, doc_path:str, obj_type, doc, id) -> DataLinkStructure:

    if hasattr(node, 'prov') and node.prov:
        meta = MetaData(page_no=node.prov[0].page_no, file_name=doc_name, location=doc_path)
    else:
         meta = MetaData(page_no=0, file_name=doc_name, location=doc_path)

    # df = node.export_to_dataframe(doc)  # In json format but data blot plot is getting high
    # data_text = df.to_dict(orient="records")
    data_text =  node.export_to_html(doc)
    return DataLinkStructure(obj_type=obj_type, level= node.level if hasattr(node, 'level') else 0, content=data_text, meta=meta, id = id)

In [7]:
from typing import cast
from docling_core.types.doc import DocItemLabel
from docling_core.types.doc.document import DoclingDocument, NodeItem

document: DoclingDocument = result.document


header_stack = {}
results = []

print(f"doc name {document_path.name}")
print(f"doc path {document_path.as_posix() }")
node: NodeItem
index: int
current_header_level:int = 0
current_header = None
doc_name = document_path.name
doc_path = document_path.as_posix()


iteration = 1
for node, index in  document.iterate_items():
    iteration = iteration+1
    if node.label == DocItemLabel.SECTION_HEADER :
        current_header = retrieve_chunk(cast(SectionHeaderItem, node), doc_name, doc_path, DocItemLabel.SECTION_HEADER, id = f"H_{iteration}")
        if (current_header.level - len(header_stack)) > 1:
            current_header.level = len(header_stack)

        previous_header_level = current_header_level
        current_header_level = 0 if current_header.level == 0 else current_header.level-1
        parent_header =  header_stack[0 if current_header_level == 0 else current_header_level-1] if header_stack and previous_header_level != current_header_level else None

        current_header.section_data = parent_header
        header_stack[current_header_level] = current_header
        if len(header_stack) > 1:
            for k in list(header_stack):
                if k > current_header_level:
                    del header_stack[k]
        #results.append(current_header)


    elif node.label == DocItemLabel.TEXT:
        text_details = retrieve_chunk(node, doc_name, doc_path, DocItemLabel.TEXT, id = f"TX_{iteration}")
        text_details.section_data = current_header
        #print(text_details)
        results.append(text_details)

    elif node.label == DocItemLabel.TABLE:
        table_details = retrieve_table_chunk(node, doc_name, doc_path, DocItemLabel.TABLE, document, id = f"TB_{iteration}")
        table_details.section_data = current_header
        print(type(node))
        results.append(table_details)

    elif node.label == DocItemLabel.LIST_ITEM:
        item_list_details = retrieve_chunk(node, doc_name, doc_path, DocItemLabel.LIST_ITEM, id = f"ITM_{iteration}")
        item_list_details.section_data = current_header
        #print(item_list_details)
        results.append(item_list_details)

# for data in results:
#     print(f"=============> {data}")




doc name Sample.pdf
doc path data/Sample.pdf
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>
<class 'docling_core.types.doc.document.TableItem'>


Check the blox plot


In [9]:
# import matplotlib.pyplot as plt
# import numpy as np
# import tiktoken
# from transformers import AutoTokenizer, TokenizersBackend
#
# enc = tiktoken.get_encoding("cl100k_base")
# tokens = enc.encode("Hello how are you ? ,")
#
# tokenizer:TokenizersBackend = AutoTokenizer.from_pretrained(
#     "BAAI/bge-small-en-v1.5"
# )
#
# #data = [len(enc.encode(doc.__str__())) for doc in results]
# data = [ len(tokenizer.encode(doc.__str__())) for doc in results]
#
# plt.boxplot(data)
# plt.title('Box Plot of chunk lengths')  # Title
# plt.xlabel('Chunk Lengths')  # Label for x-axis
# plt.ylabel('Values')  # Label for y-axis
#
# plt.show()
#
# print(f"The median chunk lenght is : {round(np.median(data),2)}")
# print(f"The average chunk lenght is : {round(np.mean(data),2)}")
# print(f"The minimum chunk lenght is : {round(np.min(data),2)}")
# print(f"The max chunk lenght is : {round(np.max(data),2)}")
# print(f"The 75th percentile chunk length is : {round(np.percentile(data, 75),2)}")
# print(f"The 25th percentile chunk length is : {round(np.percentile(data, 25),2)}")

In [10]:
#import tiktoken

# enc = tiktoken.get_encoding("cl100k_base")
# tokens = enc.encode("Hello how are you ? ,")
# print(len(tokens))
# print(tokens)


# tokenizer:TokenizersBackend = AutoTokenizer.from_pretrained(
#     "BAAI/bge-small-en-v1.5"
# )

# tk = tokenizer.encode(text="Hello how are you ? .")
# print(len(tk))
# print(tk)

# data = [ doc if len(doc.__str__().split()) > 150 else None for doc in results]
#
# for d in data:
#     if d and d.obj_type == DocItemLabel.TABLE:
#         print(d)


Use transformer logic to convert data into embedding to persist in vector database ChromaDB.


In [8]:
import chromadb
from sentence_transformers import SentenceTransformer

ids = []
embeddings = []
documents = []
metadatas = []

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

for chunk in results:

    text_to_embed = f"""
        {chunk.content}
        """

    embedding = model.encode(
            text_to_embed,
            normalize_embeddings=True
        ).tolist()

    ids.append(chunk.id)
    embeddings.append(embedding)
    documents.append(text_to_embed)
    metadatas.append(
                {
                    "section": chunk.section,
                    "sub_section": chunk.sub_section_data,
                    "page": chunk.meta.page_no,
                    "document_name": chunk.meta.file_name,
                    "document_location":chunk.meta.location
                })

client = chromadb.HttpClient(
    host="localhost",
    port=8010
)


collection = client.get_or_create_collection(
    name="pdf_chunking"
)
collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )




Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4650.20it/s]


Execute query from pdf search


In [9]:

query = "Annual Financial Statements 2025"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(results)


{'ids': [['TX_9', 'TX_20', 'ITM_4']], 'distances': [[0.97621775, 1.124306, 1.1469898]], 'embeddings': None, 'metadatas': [[{'sub_section': '', 'document_location': 'data/Sample.pdf', 'section': 'NVIDIA Announces Financial Results for Fourth Quarter and Fiscal 2025', 'document_name': 'Sample.pdf', 'page': 1}, {'document_name': 'Sample.pdf', 'document_location': 'data/Sample.pdf', 'section': 'Outlook', 'page': 2, 'sub_section': ''}, {'document_name': 'Sample.pdf', 'page': 1, 'sub_section': '', 'document_location': 'data/Sample.pdf', 'section': 'NVIDIA Announces Financial Results for Fourth Quarter and Fiscal 2025'}]], 'documents': [['\n        For fiscal 2025, revenue was $130.5 billion, up 114% from a year ago. GAAP earnings per diluted share was $2.94, up 147% from a year ago. Non-GAAP earnings per diluted share was $2.99, up 130% from a year ago.\n        ', "\n        NVIDIA's outlook for the first quarter of fiscal 2026 is as follows:\n        ", '\n        Record quarterly revenue 

In [10]:
results['documents']

[['\n        For fiscal 2025, revenue was $130.5 billion, up 114% from a year ago. GAAP earnings per diluted share was $2.94, up 147% from a year ago. Non-GAAP earnings per diluted share was $2.99, up 130% from a year ago.\n        ',
  "\n        NVIDIA's outlook for the first quarter of fiscal 2026 is as follows:\n        ",
  '\n        Record quarterly revenue of $39.3 billion, up 12% from Q3 and up 78% from a year ago\n        ']]